# Semana 13: Segmentación de clientes

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.spatial.distance import minkowski
df = pd.read_csv(BASE + "clientes_mayoristas.csv")
G = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
Z = StandardScaler().fit_transform(np.log(df[G]))

## 1. Medir distancias (RAE 1.1)

In [ ]:
a, b = Z[0], Z[1]
print("Manhattan", round(np.abs(a - b).sum(), 3), " scipy", round(minkowski(a, b, 1), 3))
print("Euclidiana", round(np.sqrt(((a - b) ** 2).sum()), 3), " scipy", round(minkowski(a, b, 2), 3))
print("Suprema", round(np.abs(a - b).max(), 3), " categoría:", G[np.abs(a - b).argmax()])
print("Coseno", round(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)), 3))

## 2. Preparar la escala

In [ ]:
print("Fresh: media", round(df["Fresh"].mean()), " mediana", df["Fresh"].median(), " máximo", df["Fresh"].max())
crudo = KMeans(2, n_init=20, random_state=42).fit(StandardScaler().fit_transform(df[G]))
km = KMeans(2, n_init=20, random_state=42).fit(Z)
print("Sin logaritmo:", sorted(np.bincount(crudo.labels_)), " con logaritmo:", sorted(np.bincount(km.labels_)))

## 3. K-means y siluetas (RAE 3.1)

In [ ]:
filas = []
for k in range(2, 9):
    m = KMeans(k, n_init=20, random_state=42).fit(Z)
    filas.append((k, round(m.inertia_, 1), round(silhouette_score(Z, m.labels_), 3)))
tabla = pd.DataFrame(filas, columns=["k", "inercia", "silueta"]); print(tabla.to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(tabla["k"], tabla["inercia"], "o-"); ax[0].set_title("Inercia: sin codo claro")
ax[1].bar(tabla["k"], tabla["silueta"]); ax[1].set_title("La silueta elige k = 2"); plt.show()

## 4. Otros métodos y validación

In [ ]:
print(pd.crosstab(km.labels_, df["Channel"], rownames=["grupo"], colnames=["canal"]))
print("K-means Rand ajustado:", round(adjusted_rand_score(df["Channel"], km.labels_), 3))
w = AgglomerativeClustering(2, linkage="ward").fit(Z)
print("Ward Rand ajustado:", round(adjusted_rand_score(df["Channel"], w.labels_), 3), " silueta", round(silhouette_score(Z, w.labels_), 3))
for e in [1.0, 1.5, 2.0]:
    lab = DBSCAN(eps=e, min_samples=8).fit(Z).labels_
    print("DBSCAN eps", e, " grupos", len(set(lab)) - (1 if -1 in lab else 0), " ruido", (lab == -1).sum())

## 5. Publicar los perfiles

In [ ]:
perfil = df.assign(grupo=km.labels_).groupby("grupo")[G].median()
perfil["clientes"] = np.bincount(km.labels_)
print(perfil)

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.